# Twitter Sentiment Analysis Exploration

## Overview
This notebook explores sentiment analysis techniques on Twitter data using various machine learning models.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Data Loading

In [2]:
# Load sample dataset
df = pd.read_csv('../data/tweets_sample_1.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst few rows:")
print(df.head())

## Exploratory Data Analysis

In [3]:
# Sentiment distribution
sentiment_counts = df['sentiment'].value_counts()
print("Sentiment Distribution:")
print(sentiment_counts)

# Plot sentiment distribution
plt.figure(figsize=(10, 6))
sentiment_counts.plot(kind='bar', color=['green', 'red', 'gray'])
plt.title('Sentiment Distribution in Twitter Dataset')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Text Preprocessing

In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

def clean_text(text):
    """Clean and preprocess text"""
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)
print("Text cleaning completed. Sample cleaned texts:")
print(df[['text', 'clean_text']].head())

## Model Training

In [5]:
# Prepare features and labels
X = df['clean_text']
y = df['sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Vectorize text
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_vec, y_train)

# Make predictions
y_pred = rf_model.predict(X_test_vec)

print("Model training completed!")

## Model Evaluation

In [6]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Feature Importance

In [7]:
# Get feature importance
feature_names = vectorizer.get_feature_names_out()
importances = rf_model.feature_importances_

# Create DataFrame of feature importances
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(20)

# Plot top features
plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance_df, x='importance', y='feature')
plt.title('Top 20 Most Important Features for Sentiment Analysis')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrates a basic sentiment analysis pipeline for Twitter data. Key findings:

1. **Data Quality**: The dataset contains balanced sentiment classes
2. **Model Performance**: Random Forest achieves good accuracy on this dataset
3. **Feature Importance**: Certain words are strong indicators of sentiment

### Next Steps:
1. Experiment with deep learning models (BERT, LSTM)
2. Incorporate additional features (emoji sentiment, hashtag analysis)
3. Deploy model as a real-time API service